# Deep Learning Building Blocks:

## Affine maps:
One of the core workhorse of deep learning is the affine maps, which is a function f(x) where,<br>
```f(x) = Ax + b```
Here, A is a matrix and vectors x and b. The parameter to be learned here A and b, b is referred to as the bias term.

https://docs.pytorch.org/tutorials/beginner/nlp/deep_learning_tutorial.html


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

torch.manual_seed(1)

In [18]:
lin = nn.Linear(3, 1)
print("Deep learning layer: ", lin)
print(lin.weight)
print(lin.bias)

data = torch.randn(2, 3)
print("\n\nInput data:", data)
print(lin(data))

Deep learning layer:  Linear(in_features=3, out_features=1, bias=True)
Parameter containing:
tensor([[-0.4591,  0.5020, -0.3513]], requires_grad=True)
Parameter containing:
tensor([-0.2258], requires_grad=True)


Input data: tensor([[-0.4699, -1.6271, -0.1127],
        [ 1.5980, -0.8445, -1.0489]])
tensor([[-0.7872],
        [-1.0148]], grad_fn=<AddmmBackward0>)


# Non linearity
First, note the following fact, which will explain why we need non-linearities in the first place. Suppose we have two
affine maps `f(x)=Ax+bf(x)=Ax+b` and `g(x)=Cx+dg(x)=Cx+d`. What is `f(g(x))f(g(x))`? <br>
`f(g(x))=A(Cx+d)+b=ACx+(Ad+b)`<br>
`f(g(x))=A(Cx+d)+b=ACx+(Ad+b)`<br>

`AC` is a matrix and `Ad+b` is a vector, so we see that composing affine maps gives you an affine map.

From this, you can see that if you wanted your neural network to be long chains of affine compositions, that this adds
no new power to your model than just doing a single affine map.

If we introduce non-linearities in between the affine layers, this is no longer the case, and we can build much more
powerful models.

There are a few core non-linearities. `tanh⁡(x),σ(x),ReLU(x)tanh(x),σ(x),ReLU(x)` are the most common.<br>
You are probably wondering: “why these functions? I can think of plenty of other non-linearities.” The reason for
this is that they have gradients that are easy to compute, and computing gradients is essential for learning.
For example
```
dσ/dx=σ(x)(1−σ(x))
```

A quick note: although you may have learned some neural networks in your intro to AI class where `σ(x)`
was the default non-linearity, typically people shy away from it in practice. This is because the gradient
vanishes very quickly as the absolute value of the argument grows. Small gradients means it is hard to
learn. Most people default to tanh or ReLU.


In [30]:
data = torch.randn(2, 2)
print("Input data:", data)
print("\n")
print("Relu: ",F.relu(data))

Input data: tensor([[ 0.6898,  0.4515],
        [-0.9094,  0.0729]])


Relu:  tensor([[0.6898, 0.4515],
        [0.0000, 0.0729]])


In [11]:
data = torch.randn(5)
print("original data:", data)
print("\n")
print(F.softmax(data, dim=0))
print(F.softmax(data, dim=0).sum())
print(F.log_softmax(data, dim=0))

original data: tensor([ 0.8310, -0.2477, -0.8029,  0.2366,  0.2857])


tensor([0.3750, 0.1275, 0.0732, 0.2069, 0.2174])
tensor(1.)
tensor([-0.9808, -2.0596, -2.6148, -1.5753, -1.5262])


# Logistic Regression Bag-Of-Words classifier

In [33]:
data = [
    ("me gusta en la cafeteria".split(), "SPANISH"),
    ("Give it to me".split(), "ENGLISH"),
    ("No creo que sea una buena idea".split(), "SPANISH"),
    ("No it is not a good idea to get lost at sea".split(), "ENGLISH"),
]
test_data = [("Yo creo que si".split(), "SPANISH"),
             ("it is lost on me".split(), "ENGLISH")]

word_to_ix = {}
for sent, _ in data + test_data:
    for word in sent:
        if word not in word_to_ix:
            word_to_ix[word] = len(word_to_ix)
print(word_to_ix)
VOCAB_SIZE = len(word_to_ix)
NUM_LABELS = 2

class BoWClassifier(nn.Module):
    def __init__(self, num_labels, vocab_size):
        super(BoWClassifier, self).__init__()

        self.linear = nn.Linear(vocab_size, num_labels)

    def forward(self, bow_vector):
        return F.log_softmax(self.linear(bow_vector), dim=1)

def make_bow_vector(sentence, word_to_ix):
    vec = torch.zeros(len(word_to_ix))
    for word in sentence:
        vec[word_to_ix[word]] = 1
    return vec.view(1, -1)

def make_target(label, label_to_ix):
    return torch.LongTensor([label_to_ix[label]])

model = BoWClassifier(num_labels=NUM_LABELS, vocab_size=VOCAB_SIZE)

for parm in model.parameters():
    print(parm)

with torch.no_grad():
    sample = data[0]
    bow_vector = make_bow_vector(sample[0], word_to_ix)
    print("make bow vector: ", bow_vector)
    log_probs = model(bow_vector)
    print("\n========", log_probs)


{'me': 0, 'gusta': 1, 'en': 2, 'la': 3, 'cafeteria': 4, 'Give': 5, 'it': 6, 'to': 7, 'Yo': 8, 'creo': 9, 'que': 10, 'si': 11, 'is': 12, 'lost': 13, 'on': 14}
Parameter containing:
tensor([[ 0.1284,  0.0798, -0.0598,  0.2489,  0.0523, -0.0666, -0.0037,  0.2538,
          0.1734, -0.0191,  0.2531,  0.1134, -0.1375, -0.2350,  0.1500],
        [ 0.2421, -0.1504,  0.1060,  0.0110, -0.0552, -0.0702,  0.0718,  0.0569,
          0.1744,  0.0945,  0.1175,  0.2278,  0.0804,  0.0383, -0.0940]],
       requires_grad=True)
Parameter containing:
tensor([0.1894, 0.0381], requires_grad=True)
make bow vector:  tensor([[1., 1., 1., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]])

======== tensor([[-0.4943, -0.9416]])


In [29]:
label_to_ix = {"SPANISH": 0, "ENGLISH": 1}
with torch.no_grad():
    for instance, label in test_data:
        bow_vec = make_bow_vector(instance, word_to_ix)
        log_probs = model(bow_vec)
        print(log_probs)

print(next(model.parameters())[:, word_to_ix['creo']])

loss_function = nn.NLLLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01, weight_decay=2, momentum=0.9)

for epoch in range(100):
    for instance, label in data:
        model.zero_grad()
        bow_vec = make_bow_vector(instance, word_to_ix)
        target = make_target(label, label_to_ix)

        log_probs = model(bow_vec)

        loss = loss_function(log_probs, target)
        loss.backward()
        optimizer.step()

with torch.no_grad():
    for instance, label in test_data:
        bow_vec = make_bow_vector(instance, word_to_ix)
        log_probs = model(bow_vec)
        print(log_probs)

# Index corresponding to Spanish goes up, English goes down!
print(next(model.parameters())[:, word_to_ix["creo"]])


tensor([[-1.1130, -0.3983]])
tensor([[-0.5530, -0.8561]])
tensor([-0.1543,  0.0318], grad_fn=<SelectBackward0>)
tensor([[-0.4823, -0.9607]])
tensor([[-1.0888, -0.4104]])
tensor([ 0.0806, -0.2030], grad_fn=<SelectBackward0>)
